In [1]:
import re
class ReActAgent:
    """
    Implement here external: websearch and Internal: RAG
    """
    def __init__(self, llm, retriever, web_search, max_iterations = 5):
        self.llm = llm
        self.retriever = retriever
        self.web_search = web_search
        self.max_iterations = max_iterations

    def think(self, query, observations):
        """
        Plain next action based on query and observation
        """
        obs_text = "\n".join([f"- {obs}" for obs in observations]) if observations else "None yet"

        system_prompt = """
        You are a healthcare assistant agent using ReAct (Reasoning + Acting).
        Plan your next action: RETRIEVE (internal knowledge) or SEARCH (web) or ANSWER (finish).

        Use RETRIEVE for: medical procedures, clinical guidelines, policy information
        Use SEARCH for: hospital hours, contacts, current announcements
        Use ANSWER when: you have enough information to respond

        Format:
        THOUGHT: <your reasoning>
        ACTION: <RETRIEVE|SEARCH|ANSWER>
        QUERY: <search query if RETRIEVE or SEARCH>
        """.strip()

        prompt = f"""
        User Question: {query}

        Previous Observations:
        {obs_text}

        What should you do next?
        """.strip()

        response = self.llm.generate(prompt=prompt, system_prompt=system_prompt)
        text = response["response"]

        # Parse response
        thought_match = re.search(r"THOUGHT:(.+?)(?=ACTION:|$)", text, re.DOTALL | re.IGNORECASE)
        action_match = re.search(r"ACTION:\s*(\w+)", text, re.IGNORECASE)
        query_match = re.search(r"QUERY:(.+)", text, re.IGNORECASE)

        thought = thought_match.group(1).strip() if thought_match else "Continue processing"
        action = action_match.group(1).strip().upper() if action_match else "ANSWER"
        action_query = query_match.group(1).strip() if query_match else query

        return {
                    "thought": thought,
                    "action": action,
                    "query": action_query,
                    "tokens": response["total_tokens"],
                    "latency_ms": response["latency_ms"]
                    }

    def act(self, action, query):
        """
        Execute Action: RAG or Web search
        """
        if action == "RETRIEVE":
            chunks = self.retriever.retrieve(query, max_k=3)
            if chunks:
                observation = f"Internal knowledge retrieved ({len(chunks)} sources):\n"
                for chunk in chunks:
                    observation += f"- [{chunk['source']}, Page {chunk.get('page', '?')}] {chunk['text'][:200]}...\n"
            else:
                observation = "No relevant internal knowledge found."
            
            return {
                "observation": observation,
                "sources": [f"{c['source']} (p.{c.get('page', '?')})" for c in chunks],
                "action_type": "retrieve"
            }
            
        elif action == "SEARCH":
            search_result = self.web_search.search(query, max_results=3)
            formatted = self.web_search.format_results(search_result)
            
            return {
                "observation": f"Web search completed:\n{formatted}",
                "sources": [r["url"] for r in search_result.get("results", [])],
                "action_type": "search"
            }
        
        else:  # ANSWER
            return {
                "observation": "Ready to provide final answer.",
                "sources": [],
                "action_type": "answer"
            }

    def answer(self, query, observations):
        """
        Generate final answer from observation
        """
        obs_text = "\n\n".join([f"Observation {i+1}:\n{obs}" for i, obs in enumerate(observations)])

        system_prompt = """
        You are a healthcare assistant providing a final answer.
        Synthesize the observations into a clear, helpful response.
        Cite sources when referencing specific information.
        Always end with: "This is educational information; verify with the hospital / consult your clinician."
        """.strip()

        prompt = f"""
        User Question: {query}

        Information Gathered:
        {obs_text}

        Provide a comprehensive final answer.
        """.strip()

        response = self.llm.generate(prompt=prompt, system_prompt=system_prompt)

        return {
            "answer": response["response"],
            "tokens": response["total_tokens"],
            "latency_ms": response["latency_ms"]
        }

    def run(self, query, verbose = True):
        """
        This will run compleate ReAct loop based on query
        """
        import time

        start_time = time.time()
        observations = []
        all_sources = []
        total_tokens = 0
        steps = []

        if verbose:
            print("=" * 80)
            print("REACT AGENT STARTING")
            print("=" * 80)
            print(f"Query: {query}\n")

        for iteration in range(self.max_iterations):
            if verbose:
                print(f"\n{'─' * 80}")
                print(f"ITERATION {iteration + 1}")
                print(f"{'─' * 80}")

            # Think
            think_result = self.think(query, observations)
            total_tokens += think_result["tokens"]

            if verbose:
                print(f"\nTHOUGHT: {think_result['thought']}")
                print(f"ACTION: {think_result['action']}")
                if think_result['action'] != 'ANSWER':
                    print(f"QUERY: {think_result['query']}")

            steps.append({
                "iteration": iteration + 1,
                "thought": think_result["thought"],
                "action": think_result["action"]
            })

            # Act
            if think_result["action"] == "ANSWER":
                break

            act_result = self.act(think_result["action"], think_result["query"])
            observations.append(act_result["observation"])
            all_sources.extend(act_result["sources"])

            if verbose:
                print(f"\nOBSERVATION:")
                print(act_result["observation"][:500] + "..." if len(act_result["observation"]) > 500 else act_result["observation"])

        # Final answer
        if verbose:
            print(f"\n{'=' * 80}")
            print("GENERATING FINAL ANSWER")
            print("=" * 80)

        answer_result = self.answer(query, observations)
        total_tokens += answer_result["tokens"]

        if verbose:
            print(f"\nANSWER:\n{answer_result['answer']}")

        end_time = time.time()
        total_time_ms = int((end_time - start_time) * 1000)

        if verbose:
            print(f"\n{'=' * 80}")
            print("SUMMARY")
            print("=" * 80)
            print(f"Iterations: {len(steps)}")
            print(f"Total tokens: {total_tokens}")
            print(f"Total time: {total_time_ms}ms")
            print(f"Sources: {len(set(all_sources))}")
        
        return {
            "query": query,
            "steps": steps,
            "observations": observations,
            "answer": answer_result["answer"],
            "sources": list(set(all_sources)),
            "total_tokens": total_tokens,
            "total_time_ms": total_time_ms
        }